# 🧠 LangGraph State Model 核心解析

> Node 返回的不是 next state，而是 **state update（delta）**。
> delta 如何变成 next state，由每个 field 的 reducer 决定。

## ✅ 普通 State（你的认知完全正确）

对于普通 field，node 返回什么，就直接 merge 覆盖对应 key。

In [ ]:
from typing import TypedDict

class MyState(TypedDict):
    foo: str
    bar: str

初始 state：

```python
{"foo": "foo", "bar": "bar"}
```

某个 node 返回：

In [ ]:
def node(state):
    return {"foo": "NEW"}

LangGraph merge 后：

```python
{"foo": "NEW", "bar": "bar"}
```

后面的 node 看到的确实就是这个。✅

---

## ⚠️ 但是 `MessagesState` 不一样！

`MessagesState` 有一个特殊的 **reducer**：

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list, add_messages]

重点是 `Annotated[..., add_messages]`

意思：

> 当 node 返回 `"messages"` 时，不要用普通 merge，要调用：

```python
add_messages(old_value, new_value)
```

所以不是：
```python
state["messages"] = returned_value
```

而是：
```python
state["messages"] = add_messages(old_messages, returned_messages)
```

这俩完全不是一回事！

---

## 🔍 实际例子：RemoveMessage 的真实行为

初始 state：

```python
state = {"messages": [1, 2, 3, 4]}
```

### filter node 返回：

```python
{"messages": [RemoveMessage(id="1"), RemoveMessage(id="2")]}
```

### ❌ 你以为的：

```python
state = {"messages": [RemoveMessage(...), RemoveMessage(...)]}
```

### ✅ 真实发生的：

LangGraph 偷偷做：

```python
new_messages = add_messages(
    old_messages=[1, 2, 3, 4],
    new_messages=[RemoveMessage("1"), RemoveMessage("2")]
)
```

---

## ⚙️ `add_messages` 内部逻辑

```python
for m in new_messages:
    if isinstance(m, RemoveMessage):
        # 删除 old_messages 中对应 id
    else:
        # append/update
```

处理过程：

| 步骤 | 操作 | 结果 |
|------|------|------|
| 1 | `RemoveMessage("1")` | `[2, 3, 4]` |
| 2 | `RemoveMessage("2")` | `[3, 4]` |

最终：
```python
state = {"messages": [3, 4]}
```

所以 `chat_model_node` 看到的是 `[3, 4]`，不是 `[RemoveMessage(...)]`！

---

## 🎯 Redux 类比（最好理解的方式）

普通 field：
```python
next_state.foo = action.payload
```

messages 更像：
```python
next_state.messages = messagesReducer(previous_state.messages, action.payload)
```

也就是：
```text
old_state + delta → new_state
```

不是：
```text
delta → new_state
```

---

## 📊 完整流程图

```text
                old state
                     │
                     ▼
      {"messages":[1,2,3,4]}
                     │
                     ▼
            filter node
                     │
                     ▼
   {"messages":[Remove(1),Remove(2)]}
                     │
                     ▼
          add_messages reducer
        (不是直接覆盖！！！)
                     │
                     ▼
      {"messages":[3,4]}
                     │
                     ▼
          chat_model_node
```

---

## 💡 核心总结

> **Node 返回的不是 next state。**
>
> **Node 返回的是 state update（delta）。**

> delta 如何变成 next state，由每个 field 的 **reducer** 决定。

这就是 LangGraph 最核心的 state model。🎯